# Diffusion Model Evaluation & Test

Evaluates **VideoContextUNet** checkpoints from `train_diffusion.ipynb` on the held-out **validation** (`EVAL_SPLIT='val'`) or **test** (`EVAL_SPLIT='test'`) split. Per-split results are written under `diffusion_eval/<split>/`.

**Architecture (VDM/FDM-inspired, conditional).** All K = k+1 frames are batch-expanded through the same UNet backbone (VDM factorized space-time backbone; Ho et al., 2022 §3), with TemporalAttnBlocks after every spatial attention block and an observed/latent mask channel (FDM; Harvey et al., NeurIPS 2022 §4). Per-frame input: `[pixel | mask | time_map]`. Conditioning is **explicit / CSDI-style** (context frames supplied clean, only the target noised) — not VDM's joint reconstruction-guided conditioning — and sampling is **autoregressive**, so expect compounding rollout drift at longer horizons.

**Residual-target models** are auto-detected (from the checkpoint config or the `_resid` run tag): sampling reconstructs `frame = newest_context + generated_residual`, then clamps to [-1, 1].

**Phase-classifier guidance is disabled** — available classifier checkpoints were trained at 256 px and are incompatible with the 128 px diffusion model.

**Environment note.** Run this notebook with the kernel that has your working **cu128 PyTorch**. FID additionally needs `torchmetrics` **and** `torch-fidelity`, installed via **pip** in that same env — installing them with `conda` can replace the pip torch build and break CUDA.


## Evaluation & Test Protocol

Use `EVAL_SPLIT = 'val'` while iterating (model selection, CFG tuning); switch to `'test'` for the final held-out numbers reported in the thesis. Each split writes to its own `diffusion_eval/<split>/` folder, so val and test artifacts never overwrite each other — run the notebook once per split.

**Studies included:**
- **One-step prediction** — MSE, L1, PSNR, SSIM, LPIPS overall and stratified by plane / phase / transition.
- **Rollout evaluation** — autoregressive H-step error for H ∈ {3, 5, 10}, with a copy-last-frame rollout reference at each horizon.
- **FID** — Fréchet Inception Distance (needs ≥ 2 048 samples; requires `torchmetrics` + `torch-fidelity`).
- **Copy-last-frame baseline** — trivial predictor for both one-step and rollout, plus an explicit "does the model beat it?" verdict table.
- **Qualitative grids** — context strip + copy-last / target / generated side-by-side (one-step, `QUALITATIVE_SAMPLES` per model) and rollout grids with aligned real / copy-last / generated rows per step (`QUALITATIVE_ROLLOUT_SAMPLES` per horizon), each panel annotated with per-frame MSE for direct visual + numeric comparison against the trivial baseline.

**Runtime/cost.** One-step + rollout + FID + the CFG sweep is heavy (DDIM × CFG forward passes at every step). Byte-identical checkpoints are de-duped automatically; set `ENABLE_FID=False` to skip the 2 048-sample FID pass while iterating.


In [8]:
from pathlib import Path
import os, math, random, glob, re, json
from contextlib import contextmanager
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from diffusers import DDPMScheduler, DDIMScheduler

import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != 'diffusion' and (PROJECT_ROOT / 'diffusion').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'diffusion'

sys.path.insert(0, str(PROJECT_ROOT))
import models as _models_module
import importlib; importlib.reload(_models_module)
from models import VideoContextUNet


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

TQDM_BAR_FORMAT = "{l_bar}{bar} {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
def tqdm_bar(iterable, **kwargs):
    return tqdm(iterable, leave=True, bar_format=TQDM_BAR_FORMAT, **kwargs)


DATA_ROOT   = PROJECT_ROOT.parent / 'data'
SILVER_ROOT = DATA_ROOT / 'embryo_dataset_silver'
MODEL_DIR   = PROJECT_ROOT / 'embryo_diffusion_models'
CACHE_DIR   = PROJECT_ROOT / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---- Evaluation settings ----
# This notebook serves BOTH validation (EVAL_SPLIT='val', for model selection / iteration)
# and the final held-out test (EVAL_SPLIT='test', for the reported numbers). Each split
# writes to its own folder diffusion_eval/<split>/, so val and test artifacts never clash.
# Run the notebook once per split.
EVAL_SPLIT = 'test'
assert EVAL_SPLIT in {'val', 'test'}

# Sample counts raised for statistical validity.  The previous run used n=6 one-step
# rows (and n=1 per phase) — far too few to distinguish the model from the
# copy-last-frame baseline (standard errors swamped the differences).
ONE_STEP_SAMPLES_PER_PLANE      = 200
TRANSITION_SAMPLES_PER_PLANE    = 100
NON_TRANSITION_SAMPLES_PER_PLANE = 100
ROLLOUT_HORIZONS                 = [3, 5, 10]
ROLLOUT_SAMPLES_PER_HORIZON      = 30
EVAL_BATCH_SIZE                  = 16
# One-step qualitative grids reuse generations already computed for the metrics pass
# (free to raise). Rollout grids require fresh DDIM sampling per sample per horizon, so
# they get their own (smaller) budget to bound added runtime.
QUALITATIVE_SAMPLES              = 20
QUALITATIVE_ROLLOUT_SAMPLES      = 20
QUALITATIVE_ROLLOUT_HORIZONS     = list(ROLLOUT_HORIZONS)
SAVE_QUALITATIVE                 = True
ENABLE_LPIPS                     = True
ENABLE_FID                       = True
FID_SAMPLES                      = 4096

# FID requires BOTH torchmetrics AND torch-fidelity (torchmetrics' Inception feature
# extractor is provided by torch-fidelity).  Check both here and FAIL LOUD instead of
# silently reporting fid=null later.  IMPORTANT: install these with PIP into the SAME
# environment as your working PyTorch — do NOT `conda install` them, as conda can replace
# the pip CUDA (cu128) torch build with an incompatible one and break the whole env.
if ENABLE_FID:
    _fid_missing = None
    try:
        import torchmetrics  # noqa: F401
        try:
            import torch_fidelity  # noqa: F401
        except Exception:
            _fid_missing = 'torch-fidelity'
    except Exception as _e:
        _fid_missing = f'torchmetrics ({type(_e).__name__}: {_e})'
    if _fid_missing is not None:
        print('=' * 78)
        print(f'WARNING: FID disabled — {_fid_missing} not importable. FID will be reported n/a.')
        print('  FID needs BOTH packages in the SAME env as your working torch:')
        print('    pip install "torchmetrics>=1.0" torch-fidelity scipy')
        print('  (use pip, NOT conda, to avoid replacing the cu128 torch build)')
        print('=' * 78)
        ENABLE_FID = False

# CFG guidance sweep (one-step only) — run the dedicated sweep cell near the end to
# find the guidance scale that minimises LPIPS/MSE.  The previous default of 3.0
# over-sharpened the near-deterministic next-frame predictions; 1.0-1.5 is expected
# to be best.  None in EVAL_CFG_SCALES means "use the checkpoint's CFG_GUIDANCE_SCALE".
EVAL_CFG_SCALES = [1.0, 1.5, 2.0, 3.0]

# DDIM steps lowered 250 -> 64.  64 steps saturates DDIM quality for this model
# (see train notebook note); 250 was ~4x wasted compute.  None = checkpoint config.
EVAL_DDIM_STEPS = 64
EVAL_DDIM_ETA   = None

ARTIFACT_ROOT    = PROJECT_ROOT / 'diffusion_eval' / EVAL_SPLIT
QUALITATIVE_ROOT = ARTIFACT_ROOT / 'qualitative'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
QUALITATIVE_ROOT.mkdir(parents=True, exist_ok=True)

METRICS_SUMMARY_PATH = ARTIFACT_ROOT / 'metrics_summary.json'
PER_PLANE_CSV_PATH   = ARTIFACT_ROOT / 'per_plane.csv'
PER_PHASE_CSV_PATH   = ARTIFACT_ROOT / 'per_phase.csv'
TRANSITION_CSV_PATH  = ARTIFACT_ROOT / 'transition_metrics.csv'
ROLLOUT_CSV_PATH     = ARTIFACT_ROOT / 'rollout_by_horizon.csv'
SUMMARY_CSV_PATH     = ARTIFACT_ROOT / 'summary.csv'

print('Artifacts root:', ARTIFACT_ROOT)
print(f'EVAL_SPLIT={EVAL_SPLIT}  DDIM_STEPS={EVAL_DDIM_STEPS}  one_step/plane={ONE_STEP_SAMPLES_PER_PLANE}')
print(f'CFG sweep scales: {EVAL_CFG_SCALES}')
print(f'FID enabled: {ENABLE_FID}  (samples={FID_SAMPLES})')


Device: cuda
Artifacts root: /home/khasion/Projects/thesis/diffusion/diffusion_eval/test
EVAL_SPLIT=test  DDIM_STEPS=64  one_step/plane=200
CFG sweep scales: [1.0, 1.5, 2.0, 3.0]
FID enabled: True  (samples=4096)


## Checkpoint Discovery and Architecture Reconstruction

Architecture parameters are read from the checkpoint's `config` dict. Parameters absent from the config (e.g. `N_ATTN_STAGES`, `TEMPORAL_ATTN_HEADS`) are inferred from the state-dict structure with sensible fallbacks so the notebook remains compatible with all checkpoints saved by `train_diffusion.ipynb`.


In [9]:
import hashlib


def strip_compile_prefix(sd: dict) -> dict:
    """Remove torch.compile '_orig_mod.' prefix from state-dict keys."""
    if not sd or not any(k.startswith('_orig_mod.') for k in sd):
        return sd
    return {k.replace('_orig_mod.', '', 1): v for k, v in sd.items()}


def _infer_block_out_channels(sd: dict, cfg: dict) -> tuple[int, ...]:
    raw = cfg.get('MODEL_BLOCK_OUT_CHANNELS') or cfg.get('block_out_channels')
    if raw is not None:
        return tuple(int(c) for c in raw)
    widths = {}
    for k, v in sd.items():
        m = re.match(r'^unet\.down_blocks\.(\d+)\.resnets\.0\.conv1\.weight$', k)
        if m:
            widths[int(m.group(1))] = int(v.shape[0])
    if not widths:
        raise KeyError('Cannot infer block_out_channels from state dict')
    return tuple(widths[i] for i in sorted(widths))


def _infer_layers_per_block(sd: dict, cfg: dict) -> int:
    raw = cfg.get('MODEL_LAYERS_PER_BLOCK')
    if raw is not None:
        return int(raw)
    ids = set()
    for k in sd:
        m = re.match(r'^unet\.down_blocks\.0\.resnets\.(\d+)\.', k)
        if m:
            ids.add(int(m.group(1)))
    return max(ids) + 1 if ids else 2


def _infer_n_attn_stages(sd: dict, cfg: dict) -> int:
    """Infer n_attn_stages from config or by counting down_blocks with attention layers."""
    raw = cfg.get('N_ATTN_STAGES')
    if raw is not None:
        return int(raw)
    attn_blocks = set()
    for k in sd:
        m = re.match(r'^unet\.down_blocks\.(\d+)\.attentions\.', k)
        if m:
            attn_blocks.add(int(m.group(1)))
    return len(attn_blocks) if attn_blocks else 3


def _file_content_hash(path: Path, chunk: int = 1 << 20) -> str:
    """MD5 of a file's bytes — used to detect byte-identical checkpoints."""
    h = hashlib.md5()
    with open(path, 'rb') as fh:
        for block in iter(lambda: fh.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def discover_model_checkpoints(model_dir: Path):
    """Locate best / one-step / rollout checkpoints from the training run.

    The training notebook copies `_best.pt` to `_best_one_step.pt` (identical val_loss
    criterion), so those two files are usually byte-identical; `_best_rollout.pt` only
    exists when rollout validation was enabled.  Byte-identical checkpoints are de-duped
    here so the (expensive) evaluation suite is not run twice on the same weights.
    """
    best_ckpts = sorted(
        [p for p in model_dir.glob('*_best.pt')
         if 'best_rollout' not in p.name and 'best_one_step' not in p.name],
        key=lambda p: p.stat().st_mtime, reverse=True,
    )
    if not best_ckpts:
        raise FileNotFoundError(f'No *_best.pt checkpoints found in {model_dir}')
    best_ckpt = best_ckpts[0]
    m = re.match(r'^(.*)_best\.pt$', best_ckpt.name)
    if not m:
        raise ValueError(f'Cannot infer run prefix from {best_ckpt.name}')
    prefix = m.group(1)

    # Candidate checkpoints, best_val first (canonical reporting name).
    candidates = [{'name': 'best_val', 'path': best_ckpt}]
    for pat, name in [
        (f'{prefix}_best_rollout.pt',  'best_rollout'),
        (f'{prefix}_best_one_step.pt', 'best_one_step'),
    ]:
        found = sorted(model_dir.glob(pat), key=lambda p: p.stat().st_mtime, reverse=True)
        if found:
            candidates.append({'name': name, 'path': found[0]})

    # De-dupe byte-identical files, keeping the first (most canonical) name.
    models, seen = [], {}
    for mc in candidates:
        digest = _file_content_hash(mc['path'])
        if digest in seen:
            print(f"  (skipping '{mc['name']}' -> byte-identical to '{seen[digest]}')")
            continue
        seen[digest] = mc['name']
        models.append(mc)
    return prefix, models


RUN_PREFIX, MODEL_CKPTS = discover_model_checkpoints(MODEL_DIR)
print('Run prefix:', RUN_PREFIX)
for mc in MODEL_CKPTS:
    print(f"  {mc['name']:20s} -> {mc['path'].name}")

# ---- Architecture config from primary checkpoint ----
_primary_ckpt = torch.load(MODEL_CKPTS[0]['path'], map_location='cpu', weights_only=False)
_primary_cfg  = _primary_ckpt.get('config', {})
_primary_sd   = strip_compile_prefix(_primary_ckpt['model'])

# Phase labels
PHASE_ALIAS = {'pHB': 'pEB', 'tHB': 'pEB'}

def normalize_phase_label(label: str) -> str:
    p = re.sub(r'\s+', '', str(label).strip())
    if p.startswith('t'):
        p = 'p' + p[1:]
    return PHASE_ALIAS.get(p, p)

PHASE_LABELS = []
for _lbl in _primary_cfg.get('PHASE_LABELS', [
    'pPB2','pPNa','pPNf','p2','p3','p4','p5','p6','p7','p8','p9+','pM','pSB','pB','pEB'
]):
    _n = normalize_phase_label(_lbl)
    if _n not in PHASE_LABELS:
        PHASE_LABELS.append(_n)
if 'pEB' not in PHASE_LABELS:
    PHASE_LABELS.append('pEB')
NUM_PHASES  = len(PHASE_LABELS)
PHASE_TO_ID = {p: i for i, p in enumerate(PHASE_LABELS)}

FOCAL_PLANES    = list(_primary_cfg.get('FOCAL_PLANES', ['F0']))
FOCAL_PLANE_SET = set(FOCAL_PLANES)
TARGET_PLANE    = str(_primary_cfg.get('TARGET_PLANE', 'F0'))

IMG_SIZE           = int(_primary_cfg.get('IMG_SIZE', 128))
TIMESTEPS          = int(_primary_cfg.get('TIMESTEPS', 1000))
CONTEXT_K          = int(_primary_cfg.get('MODEL_CONTEXT_K', _primary_cfg.get('CONTEXT_K', 4)))
MODEL_OUT_CHANNELS = int(_primary_cfg.get('MODEL_OUT_CHANNELS', 1))
MAX_CTX_DIST       = int(_primary_cfg.get('MAX_CTX_DIST', 64))
BETA_START         = float(_primary_cfg.get('BETA_START', 1e-4))
BETA_END           = float(_primary_cfg.get('BETA_END', 2e-2))
USE_V_PREDICTION   = bool(_primary_cfg.get('USE_V_PREDICTION', True))
CFG_GUIDANCE_SCALE = float(_primary_cfg.get('CFG_GUIDANCE_SCALE', 3.0))

# Detect time-map from saved weights
USE_TIME_MAP    = 'frame_emb.weight' in _primary_sd and 'frame_proj.weight' in _primary_sd
TIME_EMB_DIM    = int(_primary_sd['frame_emb.weight'].shape[1]) if USE_TIME_MAP else 8
MAX_FRAME_INDEX = int(_primary_sd['frame_emb.weight'].shape[0] - 1) if USE_TIME_MAP else 800

MODEL_BLOCK_OUT_CHANNELS = _infer_block_out_channels(_primary_sd, _primary_cfg)
MODEL_LAYERS_PER_BLOCK   = _infer_layers_per_block(_primary_sd, _primary_cfg)
N_ATTN_STAGES            = _infer_n_attn_stages(_primary_sd, _primary_cfg)
TEMPORAL_ATTN_HEADS      = int(_primary_cfg.get('TEMPORAL_ATTN_HEADS', 4))

_BETA_SCHEDULE   = str(_primary_cfg.get('BETA_SCHEDULE', 'squaredcos_cap_v2'))
_PREDICTION_TYPE = 'v_prediction' if USE_V_PREDICTION else 'epsilon'

DDIM_STEPS = int(EVAL_DDIM_STEPS if EVAL_DDIM_STEPS is not None else _primary_cfg.get('DDIM_STEPS', 64))
DDIM_ETA   = float(EVAL_DDIM_ETA if EVAL_DDIM_ETA is not None else _primary_cfg.get('DDIM_ETA', 0.0))

print(f'IMG_SIZE={IMG_SIZE}  CONTEXT_K={CONTEXT_K}  TIMESTEPS={TIMESTEPS}')
print(f'block_out_channels={MODEL_BLOCK_OUT_CHANNELS}  layers_per_block={MODEL_LAYERS_PER_BLOCK}')
print(f'n_attn_stages={N_ATTN_STAGES}  temporal_attn_heads={TEMPORAL_ATTN_HEADS}')
print(f'USE_TIME_MAP={USE_TIME_MAP}  TIME_EMB_DIM={TIME_EMB_DIM}  MAX_FRAME_INDEX={MAX_FRAME_INDEX}')
print(f'USE_V_PREDICTION={USE_V_PREDICTION}  beta_schedule={_BETA_SCHEDULE}')
print(f'DDIM_STEPS={DDIM_STEPS}  DDIM_ETA={DDIM_ETA}  CFG_GUIDANCE_SCALE={CFG_GUIDANCE_SCALE}')
print('FOCAL_PLANES:', FOCAL_PLANES)


  (skipping 'best_one_step' -> byte-identical to 'best_val')
Run prefix: nextframe_ctxk4_128px_u32-64-128-256-256_lpb1_vdm_resid
  best_val             -> nextframe_ctxk4_128px_u32-64-128-256-256_lpb1_vdm_resid_best.pt
IMG_SIZE=128  CONTEXT_K=4  TIMESTEPS=1000
block_out_channels=(32, 64, 128, 256, 256)  layers_per_block=1
n_attn_stages=3  temporal_attn_heads=4
USE_TIME_MAP=True  TIME_EMB_DIM=8  MAX_FRAME_INDEX=800
USE_V_PREDICTION=True  beta_schedule=squaredcos_cap_v2
DDIM_STEPS=64  DDIM_ETA=0.0  CFG_GUIDANCE_SCALE=1.5
FOCAL_PLANES: ['F0']


In [10]:
# ---- Residual-target detection + CFG-sweep-winner wiring (review changes) ----
# Detect whether the checkpoint was trained with residual-target prediction. Prefer the
# explicit config flag; fall back to the '_resid' run-stem tag (the training override cell
# appends it) so older checkpoints saved before the flag was persisted are still handled.
USE_RESIDUAL_PREDICTION = bool(
    _primary_cfg.get('USE_RESIDUAL_PREDICTION', '_resid' in RUN_PREFIX)
)

# Wire the CFG-sweep winner: if a prior run persisted a tuned guidance scale for this run
# (written by the CFG-sweep cell at the end of this notebook), use it as the default CFG for
# the main evaluation. Otherwise keep the checkpoint's CFG_GUIDANCE_SCALE.
_best_cfg_path = MODEL_DIR / f'{RUN_PREFIX}_best_cfg.json'
if _best_cfg_path.exists():
    try:
        _bc = json.loads(_best_cfg_path.read_text())
        _tuned = float(_bc.get('best_cfg_scale'))
        print(f'Wired tuned CFG_GUIDANCE_SCALE: {CFG_GUIDANCE_SCALE} -> {_tuned} '
              f'(from {_best_cfg_path.name})')
        CFG_GUIDANCE_SCALE = _tuned
    except Exception as _e:
        print(f'Could not read tuned CFG from {_best_cfg_path.name}: {_e}')

print(f'USE_RESIDUAL_PREDICTION={USE_RESIDUAL_PREDICTION}  CFG_GUIDANCE_SCALE={CFG_GUIDANCE_SCALE}')


Wired tuned CFG_GUIDANCE_SCALE: 1.5 -> 1.0 (from nextframe_ctxk4_128px_u32-64-128-256-256_lpb1_vdm_resid_best_cfg.json)
USE_RESIDUAL_PREDICTION=True  CFG_GUIDANCE_SCALE=1.0


## Dataset and Evaluation Split

The evaluation split is rebuilt from the **same cached split CSVs** as the training notebook, guaranteeing identical embryo-level train / val / test assignments.


In [11]:
# ---- Image loading utilities ----

def _preprocess_pil(img: Image.Image, size: int = IMG_SIZE) -> Image.Image:
    w, h = img.size
    m = min(w, h)
    img = img.crop(((w - m) // 2, (h - m) // 2, (w + m) // 2, (h + m) // 2))
    if img.size[0] != size or img.size[1] != size:
        img = img.resize((size, size), Image.BICUBIC)
    return img

def load_pil(path: str, size: int = IMG_SIZE) -> Image.Image:
    return _preprocess_pil(Image.open(path).convert('L'), size=size)

def pil_to_tensor(img: Image.Image) -> torch.Tensor:
    arr = np.array(img, dtype=np.float32) / 255.0
    return torch.from_numpy(arr * 2.0 - 1.0).unsqueeze(0)

def load_image(path: str, size: int = IMG_SIZE) -> torch.Tensor:
    return pil_to_tensor(load_pil(path, size=size))

def denorm(x: torch.Tensor) -> torch.Tensor:
    return (x.clamp(-1, 1) + 1) / 2

def _phase_name(phase_id) -> str:
    if phase_id is None:
        return '-'
    pid = int(phase_id)
    return PHASE_LABELS[pid] if 0 <= pid < len(PHASE_LABELS) else '-'


# ---- Frame index (cached) ----

INDEX_CACHE = CACHE_DIR / 'silver_all_planes_frame_index_phase_merged.csv'

def build_index_from_silver() -> pd.DataFrame:
    rows = []
    for plane in FOCAL_PLANES:
        plane_root = SILVER_ROOT / plane
        embryo_ids = sorted([d.name for d in plane_root.iterdir() if d.is_dir()])
        for eid in tqdm_bar(embryo_ids, desc=f'Indexing {plane}'):
            folder   = plane_root / eid
            ann_path = folder / f'{eid}_phases.csv'
            if not ann_path.exists():
                continue
            img_files = sorted(glob.glob(str(folder / '*.jpeg')))
            if not img_files:
                continue
            frame_nums = {}
            for f in img_files:
                mm = re.search(r'(\d+)\.jpeg$', os.path.basename(f))
                if mm:
                    frame_nums[int(mm.group(1))] = f
            phase_spans = []
            with open(ann_path, 'r', encoding='utf-8') as fh:
                for line in fh:
                    line = re.sub(r'[;,]', ' ', line.strip())
                    line = re.sub(r'\s+', ' ', line).strip()
                    mm = re.match(r'^(.*?)(\d+)\s+(\d+)$', line)
                    if not mm:
                        continue
                    pn = normalize_phase_label(mm.group(1).strip())
                    if pn in PHASE_TO_ID:
                        phase_spans.append((int(mm.group(2)), int(mm.group(3)), PHASE_TO_ID[pn]))
            for fn, fp in frame_nums.items():
                pid = next((p for s, e, p in phase_spans if s <= fn <= e), None)
                if pid is not None:
                    rows.append({'embryo_id': eid, 'plane': plane,
                                 'frame_num': fn, 'path': fp, 'phase_id': pid})
    return pd.DataFrame(rows)


if INDEX_CACHE.exists():
    index_df = pd.read_csv(INDEX_CACHE)
else:
    index_df = build_index_from_silver()
    index_df.to_csv(INDEX_CACHE, index=False)

print('Indexed frames:', len(index_df))
print('Planes in index:', sorted(index_df['plane'].unique().tolist()))


# ---- Embryo splits (must use same cache as training notebook) ----

SPLIT_SEED = 42
VAL_SPLIT  = 0.10
TEST_SPLIT = 0.10

_STEM             = f'splits_embryo_phase_ctxk{CONTEXT_K}_seed{SPLIT_SEED}_v{VAL_SPLIT}_t{TEST_SPLIT}'
TRAIN_SPLIT_CACHE = CACHE_DIR / f'{_STEM}_train.csv'
VAL_SPLIT_CACHE   = CACHE_DIR / f'{_STEM}_val.csv'
TEST_SPLIT_CACHE  = CACHE_DIR / f'{_STEM}_test.csv'


def stratified_split(ids, labels, val_frac, test_frac, seed):
    rng = np.random.RandomState(seed)
    train_ids, val_ids, test_ids = [], [], []
    by_label = {}
    for eid in ids:
        by_label.setdefault(labels[eid], []).append(eid)
    for group in by_label.values():
        group = list(group); rng.shuffle(group); n = len(group)
        nt = min(int(round(n * test_frac)), n)
        nv = min(int(round(n * val_frac)), n - nt)
        test_ids.extend(group[:nt])
        val_ids.extend(group[nt:nt + nv])
        train_ids.extend(group[nt + nv:])
    return set(train_ids), set(val_ids), set(test_ids)


if all(p.exists() for p in [TRAIN_SPLIT_CACHE, VAL_SPLIT_CACHE, TEST_SPLIT_CACHE]):
    train_df = pd.read_csv(TRAIN_SPLIT_CACHE)
    val_df   = pd.read_csv(VAL_SPLIT_CACHE)
    test_df  = pd.read_csv(TEST_SPLIT_CACHE)
else:
    ep = (index_df.groupby('embryo_id')['phase_id']
          .agg(lambda s: s.value_counts().idxmax()).to_dict())
    tr, va, te = stratified_split(list(ep), ep, VAL_SPLIT, TEST_SPLIT, SPLIT_SEED)
    train_df = pd.DataFrame({'embryo_id': list(tr)})
    val_df   = pd.DataFrame({'embryo_id': list(va)})
    test_df  = pd.DataFrame({'embryo_id': list(te)})
    train_df.to_csv(TRAIN_SPLIT_CACHE, index=False)
    val_df.to_csv(VAL_SPLIT_CACHE, index=False)
    test_df.to_csv(TEST_SPLIT_CACHE, index=False)

train_ids = set(train_df['embryo_id'].tolist())
val_ids   = set(val_df['embryo_id'].tolist())
test_ids  = set(test_df['embryo_id'].tolist())
print(f'Splits: train={len(train_ids)} | val={len(val_ids)} | test={len(test_ids)}')

split_to_ids = {'val': val_ids, 'test': test_ids}


# ---- PairDataset ----

class PairDataset:
    """Windowed next-frame evaluation dataset built from independent (embryo, plane) streams.

    Matches the split and indexing logic of train_diffusion.ipynb exactly so that
    no embryo bleeds across train / val / test boundaries.
    """

    def __init__(self, index_df, embryo_ids, split_seed, split_name, planes=None):
        self.planes = list(planes) if planes is not None else list(FOCAL_PLANES)
        self.df = index_df[
            index_df['embryo_id'].isin(set(embryo_ids)) &
            index_df['plane'].isin(set(self.planes))
        ].copy().reset_index(drop=True)
        self.rng = np.random.RandomState(
            split_seed + {'train': 0, 'val': 1, 'test': 2}.get(split_name, 0)
        )
        self.windows:                       list[dict] = []
        self.windows_by_plane:              dict[str, list[dict]] = {}
        self.transition_windows_by_plane:   dict[str, list[dict]] = {}
        self.non_transition_windows_by_plane: dict[str, list[dict]] = {}
        self.path_by_seq:  dict[str, dict[int, str]] = {}
        self.phase_by_seq: dict[str, dict[int, int]] = {}
        self._build_index()

    def _build_index(self):
        for eid, g_eid in self.df.groupby('embryo_id'):
            by_plane = {pl: g.sort_values('frame_num').reset_index(drop=True)
                        for pl, g in g_eid.groupby('plane')}
            for plane in self.planes:
                if plane not in by_plane:
                    continue
                g_plane   = by_plane[plane]
                fns       = g_plane['frame_num'].astype(int).tolist()
                fn_set    = set(fns)
                path_map  = dict(zip(fns, g_plane['path'].tolist()))
                phase_map = dict(zip(fns, g_plane['phase_id'].tolist()))

                valid = [
                    t for t in fns
                    if (t + 1) in fn_set
                    and all((t - i) in fn_set for i in range(CONTEXT_K))
                ]
                if not valid:
                    continue

                seq_key = f'{eid}|{plane}'
                self.path_by_seq[seq_key]  = path_map
                self.phase_by_seq[seq_key] = phase_map

                for t_end in valid:
                    row = {
                        'embryo_id': str(eid),
                        'seq_key':   seq_key,
                        'plane':     str(plane),
                        't_end':     int(t_end),
                        'tgt_frame': int(t_end + 1),
                    }
                    self.windows.append(row)
                    self.windows_by_plane.setdefault(plane, []).append(row)
                    ph_t, ph_tgt = phase_map.get(t_end), phase_map.get(t_end + 1)
                    is_trans = ph_t is not None and ph_tgt is not None and ph_t != ph_tgt
                    if is_trans:
                        self.transition_windows_by_plane.setdefault(plane, []).append(row)
                    else:
                        self.non_transition_windows_by_plane.setdefault(plane, []).append(row)

    def _build_row(self, row: dict) -> dict:
        sk     = str(row['seq_key'])
        plane  = str(row['plane'])
        t_end  = int(row['t_end'])
        pm     = self.path_by_seq[sk]
        phm    = self.phase_by_seq[sk]
        ctx_fns = [t_end - i for i in range(CONTEXT_K - 1, -1, -1)]
        out = dict(row)
        out['ctx_frames']     = ctx_fns
        out['ctx_paths']      = [pm[fn] for fn in ctx_fns]
        out['ctx_paths_vis']  = out['ctx_paths']
        out['ctx_frames_vis'] = ctx_fns
        out['ctx_planes_vis'] = [plane] * len(ctx_fns)
        out['phase_seq']      = [phm.get(fn) for fn in ctx_fns]
        out['phase_seq_vis']  = out['phase_seq']
        out['tgt_path']       = pm[t_end + 1]
        out['tgt_phase_id']   = phm.get(t_end + 1)
        ph_last, ph_tgt = phm.get(t_end), phm.get(t_end + 1)
        out['is_transition'] = (ph_last is not None and ph_tgt is not None and ph_last != ph_tgt)
        return out


print(f'Building eval dataset ({EVAL_SPLIT})...')
eval_dataset = PairDataset(index_df, split_to_ids[EVAL_SPLIT], SPLIT_SEED, EVAL_SPLIT)
print(f'Eval split: {EVAL_SPLIT} | total windows: {len(eval_dataset.windows)}')
for _pl in FOCAL_PLANES:
    _nt = len(eval_dataset.transition_windows_by_plane.get(_pl, []))
    _na = len(eval_dataset.windows_by_plane.get(_pl, []))
    print(f'  {_pl}: {_na} windows ({_nt} transition, {_na - _nt} non-transition)')


Indexed frames: 289675
Planes in index: ['F0']
Splits: train=565 | val=69 | test=69
Building eval dataset (test)...
Eval split: test | total windows: 28233
  F0: 28233 windows (722 transition, 27511 non-transition)


## Model Activation, Sampling and Metrics

**Sampling** replicates the training notebook exactly:
- Stride-1 `ctx_rel_idx = [k, k-1, …, 1]` built explicitly (matches training convention).
- CFG via two forward passes (`uncond=True` zeros context pixels + sets mask=0).
- Returns raw v-prediction / ε output to the DDIMScheduler with the matching `prediction_type`.


In [12]:
# ---- Model activation / deactivation ----

_active_model    = None   # VideoContextUNet on GPU
_ddim_sched      = None   # DDIMScheduler
active_ckpt_name = None


def unload_active_model():
    global _active_model, _ddim_sched, active_ckpt_name
    _active_model = None
    _ddim_sched   = None
    active_ckpt_name = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def activate_checkpoint(model_name: str, ckpt_path: Path) -> VideoContextUNet:
    """Load a VideoContextUNet checkpoint and prepare it for inference."""
    global _active_model, _ddim_sched, active_ckpt_name
    unload_active_model()

    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    sd   = strip_compile_prefix(ckpt['model'])

    model = VideoContextUNet(
        context_k          = CONTEXT_K,
        use_time_map       = USE_TIME_MAP,
        img_size           = IMG_SIZE,
        output_channels    = MODEL_OUT_CHANNELS,
        max_frame_index    = MAX_FRAME_INDEX,
        time_emb_dim       = TIME_EMB_DIM,
        block_out_channels = MODEL_BLOCK_OUT_CHANNELS,
        layers_per_block   = MODEL_LAYERS_PER_BLOCK,
        n_attn_stages      = N_ATTN_STAGES,
        temporal_attn_heads = TEMPORAL_ATTN_HEADS,
        max_ctx_dist       = MAX_CTX_DIST,
        cfg_dropout_prob   = 0.0,   # no dropout at eval time
    ).to(device)

    missing, unexpected = model.load_state_dict(sd, strict=True)
    if missing or unexpected:
        print(f'WARNING: missing={missing}, unexpected={unexpected}')

    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    # set_alpha_to_one=True matches the training-time DDIM scheduler (which uses the
    # diffusers default), so evaluation reproduces training/inference generation exactly.
    # clip_sample=False is essential: residual-target samples legitimately fall outside
    # [-1, 1] before the newest context frame is added back.
    sched = DDIMScheduler(
        num_train_timesteps = TIMESTEPS,
        beta_start          = BETA_START,
        beta_end            = BETA_END,
        beta_schedule       = _BETA_SCHEDULE,
        prediction_type     = _PREDICTION_TYPE,
        clip_sample         = False,
        set_alpha_to_one    = True,
    )

    _active_model    = model
    _ddim_sched      = sched
    active_ckpt_name = model_name

    n_params = sum(p.numel() for p in model.parameters())
    n_ta     = len(model._temporal_attns)
    print(f'Activated [{model_name}]: {ckpt_path.name}')
    print(f'  Parameters: {n_params:,} | TemporalAttnBlocks: {n_ta}')
    return model


# ---- Sampling with CFG ----

def _build_ctx_rel_idx(B: int) -> torch.Tensor:
    """Stride-1 ctx_rel_idx [B, k] matching the training convention: newest = 1, oldest = k."""
    return torch.arange(CONTEXT_K, 0, -1, device=device, dtype=torch.long).unsqueeze(0).expand(B, -1)


@torch.inference_mode()
def sample_from_model(
    shape: tuple,
    x_ctx: torch.Tensor,
    frame_idx: torch.Tensor | int | None = None,
    steps: int | None = None,
    eta: float | None = None,
    cfg_scale: float | None = None,
    x_init: torch.Tensor | None = None,
) -> torch.Tensor:
    """DDIM sampling with classifier-free guidance.

    shape     : (B, C, H, W)
    x_ctx     : (B, k, H, W) context frames in [-1, 1]  (or (k, H, W) for B=1)
    frame_idx : target frame absolute index(es) for time conditioning
    cfg_scale : guidance weight; None → uses CFG_GUIDANCE_SCALE from checkpoint config

    Residual-target prediction (USE_RESIDUAL_PREDICTION): the loop denoises in residual
    space; the newest context frame is added back after the loop, then clamped to [-1, 1].
    """
    if _active_model is None or _ddim_sched is None:
        raise RuntimeError('No checkpoint activated. Call activate_checkpoint() first.')

    n_steps = steps    if steps     is not None else DDIM_STEPS
    _eta    = eta      if eta       is not None else DDIM_ETA
    _cfg    = cfg_scale if cfg_scale is not None else CFG_GUIDANCE_SCALE

    B     = shape[0]
    x_ctx = x_ctx.to(device)
    if x_ctx.dim() == 3:
        x_ctx = x_ctx.unsqueeze(0)
    ctx_rel = _build_ctx_rel_idx(x_ctx.shape[0])

    # Build absolute time map once (shared across all DDIM timesteps)
    time_map  = None
    fi_tensor = None
    if USE_TIME_MAP and frame_idx is not None:
        if not torch.is_tensor(frame_idx):
            fi_tensor = torch.tensor([frame_idx] * B, device=device, dtype=torch.long)
        else:
            fi_tensor = frame_idx.to(device)
        fi_tensor = fi_tensor.clamp(0, MAX_FRAME_INDEX)
        time_map = _active_model.build_time_map(fi_tensor, shape[2], shape[3])
        if time_map.shape[0] == 1:
            time_map = time_map.expand(B, -1, -1, -1)

    _ddim_sched.set_timesteps(n_steps, device=device)
    x = x_init.to(device) if x_init is not None else torch.randn(shape, device=device)

    for ts in _ddim_sched.timesteps:
        t_batch = ts.unsqueeze(0).expand(B).to(device)

        # Conditional branch
        pred = _active_model(x, t_batch, x_ctx,
                             time_map=time_map, frame_idx=fi_tensor, ctx_rel_idx=ctx_rel)
        # CFG: combine with unconditional prediction (uncond=True zeros context + mask)
        if _cfg > 1.0:
            pred_u = _active_model(x, t_batch, x_ctx,
                                   time_map=time_map, frame_idx=fi_tensor,
                                   uncond=True, ctx_rel_idx=ctx_rel)
            pred = pred_u + _cfg * (pred - pred_u)

        x = _ddim_sched.step(pred, ts, x, eta=_eta).prev_sample

    # Residual-target reconstruction: add the newest context frame back, THEN clamp the
    # reconstructed frame to [-1, 1].  Do NOT clamp the raw residual (it can fall outside
    # [-1, 1]).  x_ctx[:, -1:] is the newest context frame the model conditioned on.
    if USE_RESIDUAL_PREDICTION:
        x_base = x_ctx[:, -1:]
        if x_base.shape[0] != x.shape[0]:
            x_base = x_base.expand(x.shape[0], -1, -1, -1)
        x = x_base + x

    return x.clamp(-1, 1).detach()


def _single_row_tensors(row: dict):
    ctx_ts   = [load_image(p).squeeze(0) for p in row['ctx_paths']]
    x_ctx    = torch.stack(ctx_ts, dim=0).unsqueeze(0).to(device)
    fi       = None
    if USE_TIME_MAP:
        fi = torch.tensor([min(int(row['tgt_frame']), MAX_FRAME_INDEX)],
                          dtype=torch.long, device=device)
    return x_ctx, fi


@torch.inference_mode()
def sample_single_row(row: dict) -> torch.Tensor:
    x_ctx, fi = _single_row_tensors(row)
    return sample_from_model(
        (1, MODEL_OUT_CHANNELS, IMG_SIZE, IMG_SIZE), x_ctx=x_ctx, frame_idx=fi,
        steps=DDIM_STEPS, eta=DDIM_ETA,
    )


@torch.inference_mode()
def sample_rows_batched(rows: list[dict], cfg_scale: float | None = None,
                        batch_size: int | None = None, desc: str | None = None):
    """One-step sampling over many rows in batches (used for FID generation).

    Yields generated frames [1, C, H, W] (on CPU) one per input row, in input order.
    Batching gives a large speedup over per-row sampling for the 2048-sample FID set.
    """
    bs = batch_size if batch_size is not None else EVAL_BATCH_SIZE
    rng = range(0, len(rows), bs)
    iterator = tqdm_bar(rng, desc=desc) if desc else rng
    for start in iterator:
        batch = rows[start:start + bs]
        tensors = rows_to_tensors(batch)
        if USE_TIME_MAP:
            x_ctx, _x_tgt, fi = tensors; fi = fi.to(device)
        else:
            x_ctx, _x_tgt = tensors; fi = None
        x_ctx = x_ctx.to(device)
        x_gen = sample_from_model(
            (x_ctx.size(0), MODEL_OUT_CHANNELS, IMG_SIZE, IMG_SIZE),
            x_ctx=x_ctx, frame_idx=fi, cfg_scale=cfg_scale,
        )
        for i in range(len(batch)):
            yield x_gen[i:i+1].cpu()


# ---- Perceptual metrics ----

_lpips_metric = None
if ENABLE_LPIPS:
    try:
        from lpips import LPIPS
        _lpips_metric = LPIPS(net='alex').to(device)
        _lpips_metric.eval()
        for _p in _lpips_metric.parameters():
            _p.requires_grad_(False)
        print('LPIPS enabled')
    except Exception as exc:
        print('LPIPS disabled:', exc)

_fid_metric = None
if ENABLE_FID:
    try:
        from torchmetrics.image.fid import FrechetInceptionDistance
        _fid_metric = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
        _fid_metric.eval()
        for _p in _fid_metric.parameters():
            _p.requires_grad_(False)
        print('FID enabled')
    except Exception as exc:
        print('FID disabled:', exc)

_gk_cache: dict = {}

def _gaussian_kernel(ws, sigma, dev, dtype):
    key = (ws, sigma, dev.type, dev.index, dtype)
    if key not in _gk_cache:
        c = torch.arange(ws, device=dev, dtype=dtype) - ws // 2
        g = torch.exp(-(c ** 2) / (2 * sigma ** 2)); g /= g.sum()
        k2d = g.unsqueeze(0).t() @ g.unsqueeze(0); k2d /= k2d.sum()
        _gk_cache[key] = k2d.unsqueeze(0).unsqueeze(0)
    return _gk_cache[key]


def compute_ssim(pred: torch.Tensor, tgt: torch.Tensor, ws: int = 11, sigma: float = 1.5) -> torch.Tensor:
    p = ((pred.clamp(-1, 1) + 1) / 2).float()
    t = ((tgt.clamp(-1, 1) + 1) / 2).float()
    C   = p.size(1)
    pad = ws // 2
    k   = _gaussian_kernel(ws, sigma, p.device, p.dtype).expand(C, 1, ws, ws)
    mp  = F.conv2d(p, k, padding=pad, groups=C)
    mt  = F.conv2d(t, k, padding=pad, groups=C)
    sp  = (F.conv2d(p * p, k, padding=pad, groups=C) - mp ** 2).clamp_min(0)
    st  = (F.conv2d(t * t, k, padding=pad, groups=C) - mt ** 2).clamp_min(0)
    spt = F.conv2d(p * t, k, padding=pad, groups=C) - mp * mt
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    return ((2 * mp * mt + c1) * (2 * spt + c2) /
            ((mp ** 2 + mt ** 2 + c1) * (sp + st + c2) + 1e-6)).mean()


def compute_lpips(pred: torch.Tensor, tgt: torch.Tensor) -> float | None:
    if _lpips_metric is None:
        return None
    def _to3(x):
        return x.repeat(1, 3, 1, 1) if x.shape[1] == 1 else x
    with torch.no_grad():
        return float(_lpips_metric(_to3(pred.clamp(-1, 1)),
                                   _to3(tgt.clamp(-1, 1)), normalize=False).mean().item())


def _to_fid_input(x: torch.Tensor) -> torch.Tensor:
    x = x.clamp(-1, 1).float()
    x = (x + 1) / 2
    return x.repeat(1, 3, 1, 1) if x.shape[1] == 1 else x


def compute_fid(real_list: list[torch.Tensor], fake_list: list[torch.Tensor],
                batch_size: int | None = None) -> float | None:
    if _fid_metric is None or min(len(real_list), len(fake_list)) < 2:
        return None
    bs = batch_size if batch_size is not None else EVAL_BATCH_SIZE
    _fid_metric.reset()

    def _update(tensors: list[torch.Tensor], real: bool):
        # Batch the Inception feature updates — far faster than one image at a time
        # for the 2048-sample FID set.
        for start in range(0, len(tensors), bs):
            chunk = torch.cat(
                [x if x.dim() == 4 else x.unsqueeze(0) for x in tensors[start:start + bs]],
                dim=0,
            )
            _fid_metric.update(_to_fid_input(chunk.to(device)), real=real)

    _update(real_list, True)
    _update(fake_list, False)
    try:
        return float(_fid_metric.compute().item())
    except Exception as exc:
        print(f'FID failed: {exc}')
        return None


def metric_record(x_gen: torch.Tensor, x_tgt: torch.Tensor) -> dict:
    x_gen = x_gen.detach(); x_tgt = x_tgt.detach()
    mse = float((x_gen - x_tgt).pow(2).mean().item())
    # PSNR in standard [0,1] image space: mapping [-1,1]->[0,1] divides MSE by 4.
    # Capped at 100 dB so identical frames don't yield +inf and corrupt averages.
    mse01 = mse / 4.0
    psnr = 100.0 if mse01 <= 1e-12 else min(100.0, float(-10.0 * math.log10(mse01)))
    return {
        'mse':   mse,
        'l1':    float((x_gen - x_tgt).abs().mean().item()),
        'psnr':  psnr,
        'ssim':  float(compute_ssim(x_gen, x_tgt).item()),
        'lpips': compute_lpips(x_gen, x_tgt),
    }


def rows_to_tensors(rows: list[dict]):
    ctx_list, tgt_list, fi_list = [], [], []
    for row in rows:
        ctx_ts = [load_image(p).squeeze(0) for p in row['ctx_paths']]
        ctx_list.append(torch.stack(ctx_ts, dim=0))
        tgt_list.append(load_image(row['tgt_path']))
        if USE_TIME_MAP:
            fi_list.append(min(int(row['tgt_frame']), MAX_FRAME_INDEX))
    x_ctx = torch.stack(ctx_list, dim=0)
    x_tgt = torch.stack(tgt_list, dim=0)
    if USE_TIME_MAP:
        return x_ctx, x_tgt, torch.tensor(fi_list, dtype=torch.long)
    return x_ctx, x_tgt


def summarize_records(records: list[dict], group_cols: list[str]) -> pd.DataFrame:
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    mc = [c for c in ['mse', 'l1', 'psnr', 'ssim', 'lpips'] if c in df.columns]
    if not group_cols:
        r = {'n': len(df)}
        for c in mc:
            s = pd.to_numeric(df[c], errors='coerce').dropna()
            r[c]           = float(s.mean()) if len(s) else float('nan')
            r[f'{c}_se']   = float(s.std() / len(s) ** 0.5) if len(s) > 1 else float('nan')
        return pd.DataFrame([r])
    rows_out = []
    for keys, grp in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        r = dict(zip(group_cols, keys)); r['n'] = len(grp)
        for c in mc:
            s = pd.to_numeric(grp[c], errors='coerce').dropna()
            r[c]         = float(s.mean()) if len(s) else float('nan')
            r[f'{c}_se'] = float(s.std() / len(s) ** 0.5) if len(s) > 1 else float('nan')
        rows_out.append(r)
    return pd.DataFrame(rows_out)


print('Model + sampling + metrics helpers defined.')


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/home/khasion/miniconda3/envs/thesis/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/khasion/miniconda3/envs/thesis/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/khasion/miniconda3/envs/thesis/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth
LPIPS enabled
FID enabled
Model + sampling + metrics helpers defined.


## Visualization Helpers

- **One-step grid**: context frames as columns on top; below, a 3-column strip — **copy-last** (frozen newest context frame) / **target** / **generated** — each labelled with its MSE against the target, so it's immediately visible whether a given sample beats the trivial baseline.
- **Rollout grid**: context strip on top; below, aligned per-step columns showing **real** ground truth, **copy-last** (frozen baseline), and **generated** frames, each with per-step MSE, so rollout drift can be visually compared against both the ground truth and the baseline at every horizon step.

Ground-truth phase labels from the dataset are overlaid on context frames. No classifier is used for generated frames.


In [13]:
def _save_sample_grid(
    rows: list[dict],
    out_dir: Path,
    title_suffix: str,
    precomputed_gen: list[torch.Tensor] | None = None,
):
    """Save one-step sample grids: context columns on top, target / generated below."""
    if not rows:
        return
    out_dir.mkdir(parents=True, exist_ok=True)

    for idx, row in enumerate(rows):
        ctx_paths_vis = row.get('ctx_paths_vis', row['ctx_paths'])
        ctx_tensors   = [load_image(p).squeeze(0) for p in ctx_paths_vis]
        tgt_tensor    = load_image(row['tgt_path'])
        target_plane  = row.get('plane', TARGET_PLANE)

        if precomputed_gen is not None and idx < len(precomputed_gen):
            x_gen = precomputed_gen[idx].to(device)
        else:
            x_ctx = torch.stack([load_image(p).squeeze(0) for p in row['ctx_paths']],
                                 dim=0).unsqueeze(0).to(device)
            fi = None
            if USE_TIME_MAP:
                fi = torch.tensor([min(int(row['tgt_frame']), MAX_FRAME_INDEX)],
                                   device=device, dtype=torch.long)
            x_gen = sample_from_model(
                (1, MODEL_OUT_CHANNELS, IMG_SIZE, IMG_SIZE), x_ctx=x_ctx, frame_idx=fi,
            )

        ctx_frames = row.get('ctx_frames_vis', row.get('ctx_frames', [None] * len(ctx_tensors)))
        ctx_phases = row.get('phase_seq_vis', row.get('phase_seq', [None] * len(ctx_tensors)))
        ctx_planes = row.get('ctx_planes_vis', [row.get('plane', TARGET_PLANE)] * len(ctx_tensors))
        frame_order = list(dict.fromkeys(ctx_frames))
        plane_order = list(dict.fromkeys(ctx_planes)) or [target_plane]

        ctx_lookup = {}
        for i, ct in enumerate(ctx_tensors):
            fv = ctx_frames[i] if i < len(ctx_frames) else None
            pn = ctx_planes[i] if i < len(ctx_planes) else None
            ph = ctx_phases[i] if i < len(ctx_phases) else None
            if fv is not None and pn is not None:
                ctx_lookup[(fv, pn)] = (ct, ph)

        n_rows = max(1, len(plane_order))
        n_cols = max(1, len(frame_order))
        fig = plt.figure(
            figsize=(min(max(2.1 * n_cols + 3.5, 10.0), 30.0),
                     min(max(1.7 * n_rows + 3.0, 7.0), 26.0)),
            constrained_layout=True,
        )
        outer    = fig.add_gridspec(2, 1, height_ratios=[n_rows, 1.2], hspace=0.18)
        ctx_grid = outer[0].subgridspec(n_rows, n_cols, wspace=0.02, hspace=0.02)

        for ri, pname in enumerate(plane_order):
            for ci, fval in enumerate(frame_order):
                ax = fig.add_subplot(ctx_grid[ri, ci])
                entry = ctx_lookup.get((fval, pname))
                if entry is None:
                    ax.axis('off'); continue
                ct, ph = entry
                ax.imshow(denorm(ct.unsqueeze(0).unsqueeze(0))[0, 0].cpu().numpy(), cmap='gray')
                if ri == 0:
                    ax.set_title(f't={fval}', fontsize=9)
                if ci == 0:
                    ax.set_ylabel(str(pname), fontsize=9)
                ax.text(0.02, 0.04, _phase_name(ph), transform=ax.transAxes, fontsize=7,
                        color='white', bbox=dict(facecolor='black', alpha=0.45, edgecolor='none', pad=1.2))
                ax.set_xticks([]); ax.set_yticks([])

        # Copy-last (frozen newest context frame) shown alongside target/generated so the
        # model's output can be visually triangulated against the same trivial baseline
        # used in the quantitative "beats copy-last?" verdict table.
        pred_grid = outer[1].subgridspec(1, 3, wspace=0.12)

        tgt_hw    = tgt_tensor.squeeze(0)          # [H, W]
        copy_last = ctx_tensors[-1]                # newest context frame (model's residual base)
        gen_hw    = x_gen[0, 0].detach().cpu()
        mse_copy  = float((copy_last - tgt_hw).pow(2).mean().item())
        mse_gen   = float((gen_hw - tgt_hw).pow(2).mean().item())
        verdict   = 'better' if mse_gen < mse_copy else 'worse'

        ax_copy = fig.add_subplot(pred_grid[0, 0])
        ax_copy.imshow(denorm(copy_last.unsqueeze(0).unsqueeze(0))[0, 0].cpu().numpy(), cmap='gray')
        ax_copy.set_title(f"Copy-last | MSE={mse_copy:.4f}")
        ax_copy.set_xticks([]); ax_copy.set_yticks([])

        ax_tgt = fig.add_subplot(pred_grid[0, 1])
        ax_tgt.imshow(denorm(tgt_tensor.unsqueeze(0))[0, 0].cpu().numpy(), cmap='gray')
        ax_tgt.set_title(
            f"Target {target_plane} | {_phase_name(row.get('tgt_phase_id'))} | t={row.get('tgt_frame', '-')}"
        )
        ax_tgt.set_xticks([]); ax_tgt.set_yticks([])

        ax_gen = fig.add_subplot(pred_grid[0, 2])
        ax_gen.imshow(denorm(x_gen)[0, 0].detach().cpu().numpy(), cmap='gray')
        ax_gen.set_title(f"Generated | MSE={mse_gen:.4f} ({verdict} than copy-last)")
        ax_gen.set_xticks([]); ax_gen.set_yticks([])

        fig.suptitle(
            f"{row['embryo_id']} ({row['plane']}) — context | {title_suffix}",
            fontsize=11,
        )
        fig.savefig(out_dir / f'sample_{idx:02d}.png', dpi=150)
        plt.close(fig)


def _run_rollout(
    ctx_tensors: list[torch.Tensor],
    t_end: int,
    steps: int,
    phase_seq: list | None = None,
) -> list[dict]:
    """Autoregressively generate `steps` future frames.

    Returns list of dicts with keys 't_frame', 'image'.
    No phase classifier is used; phase_seq is carried forward from ground truth only.
    """
    expected_k = int(getattr(_active_model, 'context_k', CONTEXT_K))
    if len(ctx_tensors) != expected_k:
        raise ValueError(f'Need exactly {expected_k} context tensors, got {len(ctx_tensors)}')

    current_ctx = torch.stack(
        [t.squeeze(0) if t.dim() == 3 else t for t in ctx_tensors], dim=0
    ).unsqueeze(0).to(device)
    current_t_end = int(t_end)
    generated = []

    for _ in range(max(0, int(steps))):
        fi = None
        if USE_TIME_MAP:
            fi = torch.tensor([min(current_t_end + 1, MAX_FRAME_INDEX)],
                              device=device, dtype=torch.long)
        x_gen = sample_from_model(
            (1, MODEL_OUT_CHANNELS, IMG_SIZE, IMG_SIZE), current_ctx, frame_idx=fi,
        )
        gen_cpu = x_gen[0, 0].detach().cpu()
        generated.append({'t_frame': current_t_end + 1, 'image': gen_cpu})
        current_ctx = torch.cat([current_ctx[:, 1:], x_gen], dim=1)
        current_t_end += 1

    return generated


def _save_rollout_grid(row: dict, out_dir: Path, steps: int):
    """Save rollout figure: context strip on top; below it, aligned per-step columns
    showing the REAL ground-truth frame, the frozen copy-last-frame baseline, and the
    model's generated frame — each annotated with per-step MSE so drift is directly
    comparable to the trivial baseline used in the quantitative rollout metrics.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    ctx_tensors = [load_image(p) for p in row['ctx_paths']]
    generated   = _run_rollout(ctx_tensors, int(row['t_end']), steps,
                               phase_seq=row.get('phase_seq'))

    ctx_frames  = list(row.get('ctx_frames_vis', row.get('ctx_frames', [])))
    ctx_phases  = list(row.get('phase_seq_vis',  row.get('phase_seq',  [])))
    ctx_images  = [load_image(p).squeeze(0) for p in row.get('ctx_paths_vis', row['ctx_paths'])]

    n_ctx     = max(1, len(ctx_images))
    n_steps   = max(1, len(generated))
    copy_last = ctx_images[-1]   # frozen newest context frame, same baseline as the metrics

    real_tensors = None
    real_paths = row.get('rollout_tgt_paths')
    if real_paths:
        real_tensors = [load_image(p).squeeze(0) for p in real_paths[:n_steps]]

    n_bottom_rows = 3 if real_tensors else 2

    fig = plt.figure(
        figsize=(min(max(2.2 * max(n_ctx, n_steps) + 2.0, 10.0), 32.0),
                 min(max(2.4 * (1 + n_bottom_rows) + 1.8, 7.0), 20.0)),
        constrained_layout=True,
    )
    outer = fig.add_gridspec(2, 1,
                             height_ratios=[1.0, max(1.2, float(n_bottom_rows))],
                             hspace=0.14)

    ctx_g = outer[0].subgridspec(1, n_ctx, wspace=0.03, hspace=0.03)
    for i, ct in enumerate(ctx_images):
        ax = fig.add_subplot(ctx_g[0, i])
        ax.imshow(denorm(ct.unsqueeze(0).unsqueeze(0))[0, 0].cpu().numpy(), cmap='gray')
        if i < len(ctx_frames):
            ax.set_title(f't={int(ctx_frames[i])}', fontsize=9)
        ph = ctx_phases[i] if i < len(ctx_phases) else None
        ax.text(0.02, 0.04, _phase_name(ph), transform=ax.transAxes, fontsize=7,
                color='white', bbox=dict(facecolor='black', alpha=0.45, edgecolor='none', pad=1.2))
        ax.set_xticks([]); ax.set_yticks([])

    bottom_g = outer[1].subgridspec(n_bottom_rows, n_steps, wspace=0.04, hspace=0.16)
    r = 0

    if real_tensors:
        for i in range(n_steps):
            ax = fig.add_subplot(bottom_g[r, i])
            if i < len(real_tensors):
                ax.imshow(denorm(real_tensors[i].unsqueeze(0).unsqueeze(0))[0, 0].cpu().numpy(), cmap='gray')
            ax.set_title(f"Real t={int(generated[i]['t_frame'])}", fontsize=9)
            if i == 0:
                ax.set_ylabel('Real', fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        r += 1

    for i in range(n_steps):
        ax = fig.add_subplot(bottom_g[r, i])
        ax.imshow(denorm(copy_last.unsqueeze(0).unsqueeze(0))[0, 0].cpu().numpy(), cmap='gray')
        if real_tensors and i < len(real_tensors):
            mse_c = float((copy_last - real_tensors[i]).pow(2).mean().item())
            ax.set_title(f"Copy-last | MSE={mse_c:.4f}", fontsize=8)
        else:
            ax.set_title("Copy-last", fontsize=9)
        if i == 0:
            ax.set_ylabel('Copy-last', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    r += 1

    for i in range(n_steps):
        ax = fig.add_subplot(bottom_g[r, i])
        gen_hw = generated[i]['image']
        ax.imshow(denorm(gen_hw.unsqueeze(0).unsqueeze(0))[0, 0].cpu().numpy(), cmap='gray')
        if real_tensors and i < len(real_tensors):
            mse_g = float((gen_hw - real_tensors[i]).pow(2).mean().item())
            ax.set_title(f"Gen t={int(generated[i]['t_frame'])} | MSE={mse_g:.4f}", fontsize=8)
        else:
            ax.set_title(f"Gen t={int(generated[i]['t_frame'])}", fontsize=9)
        if i == 0:
            ax.set_ylabel('Generated', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

    row_plane = row.get('plane', TARGET_PLANE)
    fig.suptitle(
        f"{row['embryo_id']} ({row_plane}) — rollout H={steps} (real vs copy-last vs generated)",
        fontsize=11,
    )
    out_path = out_dir / f"{row['embryo_id']}_{row_plane}_t{int(row['t_end']):05d}_rollout_h{steps}.png"
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


print('Visualization helpers defined.')


Visualization helpers defined.


## Evaluation Row Collection and Main Evaluation Suite

Fixed row sets are sampled once and reused across all checkpoints to keep the comparison fair. The evaluation loop activates each checkpoint in turn, runs all studies, then unloads the model to free VRAM before moving to the next checkpoint.


In [14]:
# ---- Row collection helpers ----

def _shuffle_rows(rows: list[dict], rng: np.random.RandomState) -> list[dict]:
    if not rows:
        return []
    order = rng.permutation(len(rows)).tolist()
    return [rows[int(i)] for i in order]


def _round_robin_take(buckets: dict[str, list[dict]], total: int) -> list[dict]:
    selected = []
    plane_order = [pl for pl in FOCAL_PLANES if buckets.get(pl)]
    while len(selected) < total:
        added = False
        for pl in plane_order:
            if not buckets.get(pl):
                continue
            selected.append(buckets[pl].pop())
            added = True
            if len(selected) >= total:
                break
        if not added:
            break
    return selected


def collect_one_step_rows(
    dataset: PairDataset,
    mode: str,
    seed: int,
    *,
    total: int | None = None,
    per_plane: int | None = None,
) -> list[dict]:
    """Sample evaluation rows for one-step prediction.

    mode: 'any' | 'transition' | 'non_transition'
    """
    if total is None and per_plane is None:
        raise ValueError('Specify total or per_plane')
    rng = np.random.RandomState(seed)
    buckets: dict[str, list[dict]] = {}
    for pl in FOCAL_PLANES:
        if mode == 'transition':
            pool = dataset.transition_windows_by_plane.get(pl, [])
        elif mode == 'non_transition':
            pool = dataset.non_transition_windows_by_plane.get(pl, [])
        else:
            pool = dataset.windows_by_plane.get(pl, [])
        rows = [dataset._build_row(r) for r in pool]
        rows = _shuffle_rows(rows, rng)
        if per_plane is not None:
            rows = rows[:per_plane]
        buckets[pl] = rows
    if total is None:
        out = []
        for pl in FOCAL_PLANES:
            out.extend(buckets.get(pl, []))
        return out
    return _round_robin_take(buckets, total)


def _build_rollout_row(
    dataset: PairDataset, base_row: dict, steps: int
) -> dict | None:
    """Extend a base row with rollout targets if all future frames exist."""
    sk    = str(base_row['seq_key'])
    t_end = int(base_row['t_end'])
    pm    = dataset.path_by_seq.get(sk, {})
    phm   = dataset.phase_by_seq.get(sk, {})
    future = [t_end + s for s in range(1, steps + 1)]
    if any(fn not in pm for fn in future):
        return None
    row = dataset._build_row(base_row)
    row['rollout_tgt_frames']   = future
    row['rollout_tgt_paths']    = [pm[fn] for fn in future]
    row['rollout_tgt_phase_ids'] = [phm.get(fn) for fn in future]
    return row


def collect_rollout_rows(
    dataset: PairDataset, steps: int, total: int, seed: int, mode: str = 'any'
) -> list[dict]:
    rng = np.random.RandomState(seed + 1000 + int(steps))
    buckets: dict[str, list[dict]] = {}
    for pl in FOCAL_PLANES:
        if mode == 'transition':
            pool = dataset.transition_windows_by_plane.get(pl, [])
        elif mode == 'non_transition':
            pool = dataset.non_transition_windows_by_plane.get(pl, [])
        else:
            pool = dataset.windows_by_plane.get(pl, [])
        candidates = [r for b in pool for r in [_build_rollout_row(dataset, b, steps)] if r]
        buckets[pl] = _shuffle_rows(candidates, rng)
    return _round_robin_take(buckets, total)


# ---- Per-study evaluation functions ----

@torch.inference_mode()
def eval_one_step_rows(
    rows: list[dict],
    desc: str,
    return_generated: bool = False,
) -> list[dict] | tuple[list[dict], list[torch.Tensor]]:
    records, generated_out = [], []
    for start in tqdm_bar(range(0, len(rows), EVAL_BATCH_SIZE), desc=desc):
        batch = rows[start:start + EVAL_BATCH_SIZE]
        tensors = rows_to_tensors(batch)
        if USE_TIME_MAP:
            x_ctx, x_tgt, fi = tensors
            fi = fi.to(device)
        else:
            x_ctx, x_tgt = tensors; fi = None
        x_ctx = x_ctx.to(device); x_tgt = x_tgt.to(device)
        x_gen = sample_from_model(
            (x_ctx.size(0), MODEL_OUT_CHANNELS, IMG_SIZE, IMG_SIZE),
            x_ctx=x_ctx, frame_idx=fi,
        )
        for i, row in enumerate(batch):
            rec = metric_record(x_gen[i:i+1], x_tgt[i:i+1])
            rec.update({
                'embryo_id':   row['embryo_id'],
                'plane':       row['plane'],
                'phase_id':    row.get('tgt_phase_id'),
                'phase_name':  PHASE_LABELS[int(row['tgt_phase_id'])] if row.get('tgt_phase_id') is not None else None,
                'is_transition': bool(row.get('is_transition')) if row.get('is_transition') is not None else None,
            })
            records.append(rec)
            if return_generated:
                generated_out.append(x_gen[i:i+1].cpu())
    if return_generated:
        return records, generated_out
    return records


@torch.inference_mode()
def eval_rollout_rows(rows: list[dict], steps: int, desc: str) -> list[dict]:
    """Autoregressive H-step rollout error, BATCHED across rows.

    All rows share the same horizon `steps`, so they advance in lockstep: at each step the
    whole batch is sampled at once and each row's context window is shifted by its own
    generated frame. This is far faster than the previous per-row loop while producing
    identical per-(row, step) metrics.
    """
    records = []
    for start in tqdm_bar(range(0, len(rows), EVAL_BATCH_SIZE), desc=desc):
        batch = rows[start:start + EVAL_BATCH_SIZE]
        # Build batched starting context [N, k, H, W] and per-row running frame index.
        ctx_list = [torch.stack([load_image(p).squeeze(0) for p in r['ctx_paths']], dim=0)
                    for r in batch]
        current_ctx = torch.stack(ctx_list, dim=0).to(device)
        t_end_cur   = torch.tensor([int(r['t_end']) for r in batch],
                                   dtype=torch.long, device=device)

        for step_i in range(1, steps + 1):
            fi = None
            if USE_TIME_MAP:
                fi = (t_end_cur + 1).clamp(max=MAX_FRAME_INDEX)
            x_gen = sample_from_model(
                (current_ctx.size(0), MODEL_OUT_CHANNELS, IMG_SIZE, IMG_SIZE),
                x_ctx=current_ctx, frame_idx=fi,
            )
            x_tgt = torch.stack(
                [load_image(r['rollout_tgt_paths'][step_i - 1]) for r in batch], dim=0
            ).to(device)
            for j, row in enumerate(batch):
                rec = metric_record(x_gen[j:j+1], x_tgt[j:j+1])
                rec.update({
                    'embryo_id':       row['embryo_id'],
                    'plane':           row['plane'],
                    'horizon':         int(steps),
                    'step':            int(step_i),
                    'seed_transition': bool(row.get('is_transition')) if row.get('is_transition') is not None else None,
                })
                records.append(rec)
            current_ctx = torch.cat([current_ctx[:, 1:], x_gen.detach()], dim=1)
            t_end_cur  = t_end_cur + 1
    return records


def eval_copy_last_frame(rows: list[dict], desc: str = 'copy-last-frame baseline') -> list[dict]:
    records = []
    for row in tqdm_bar(rows, desc=desc):
        x_pred = load_image(row['ctx_paths'][-1]).unsqueeze(0).to(device)
        x_tgt  = load_image(row['tgt_path']).unsqueeze(0).to(device)
        rec    = metric_record(x_pred, x_tgt)
        rec.update({
            'embryo_id':   row['embryo_id'],
            'plane':       row['plane'],
            'phase_id':    row.get('tgt_phase_id'),
            'is_transition': bool(row.get('is_transition')) if row.get('is_transition') is not None else None,
        })
        records.append(rec)
    return records


def eval_copy_last_rollout_rows(rows: list[dict], steps: int,
                                desc: str = 'copy-last rollout') -> list[dict]:
    """Copy-last-frame rollout baseline: predict the frozen newest context frame for all
    H steps.  Provides a 'do-nothing' rollout-drift reference at each horizon — the model
    must beat this to add value beyond simply repeating the last observed frame.
    """
    records = []
    for row in tqdm_bar(rows, desc=desc):
        x_pred = load_image(row['ctx_paths'][-1]).unsqueeze(0).to(device)  # frozen newest frame
        for step_i in range(1, steps + 1):
            x_tgt = load_image(row['rollout_tgt_paths'][step_i - 1]).unsqueeze(0).to(device)
            rec = metric_record(x_pred, x_tgt)
            rec.update({'embryo_id': row['embryo_id'], 'plane': row['plane'],
                        'horizon': int(steps), 'step': int(step_i)})
            records.append(rec)
    return records


# ---- Fixed row sets (sampled once, reused across checkpoints) ----

one_step_rows = collect_one_step_rows(
    eval_dataset, mode='any', per_plane=ONE_STEP_SAMPLES_PER_PLANE, seed=SEED + 10,
)
transition_rows = collect_one_step_rows(
    eval_dataset, mode='transition', per_plane=TRANSITION_SAMPLES_PER_PLANE, seed=SEED + 20,
)
non_transition_rows = collect_one_step_rows(
    eval_dataset, mode='non_transition', per_plane=NON_TRANSITION_SAMPLES_PER_PLANE, seed=SEED + 30,
)
rollout_rows_by_horizon = {
    h: collect_rollout_rows(eval_dataset, steps=h,
                            total=ROLLOUT_SAMPLES_PER_HORIZON,
                            seed=SEED + 100 + h)
    for h in ROLLOUT_HORIZONS
}
fid_rows = collect_one_step_rows(
    eval_dataset, mode='any', total=FID_SAMPLES, seed=SEED + 400,
) if ENABLE_FID else []

print('One-step rows:',        len(one_step_rows))
print('Transition rows:',      len(transition_rows))
print('Non-transition rows:',  len(non_transition_rows))
for h, rows in rollout_rows_by_horizon.items():
    print(f'Rollout H={h}: {len(rows)} rows')
print(f'FID rows: {len(fid_rows)} (FID unreliable below ~2048 samples)')

# Pre-load real target tensors for FID (reused for baseline + all models)
fid_real_tensors: list[torch.Tensor] = []
if ENABLE_FID and _fid_metric is not None and fid_rows:
    fid_real_tensors = [load_image(r['tgt_path']) for r in fid_rows]

# ---- Copy-last-frame baseline ----
print('\n=== Copy-last-frame baseline ===')
baseline_records    = eval_copy_last_frame(one_step_rows)
baseline_overall_df = summarize_records(baseline_records, [])
baseline_per_plane  = summarize_records(baseline_records, ['plane'])
baseline_trans_df   = summarize_records(
    [dict(r, subset='transition' if r.get('is_transition') else 'non_transition')
     for r in baseline_records], ['subset'],
)
print('Baseline overall:');    display(baseline_overall_df)
print('Baseline per-plane:');  display(baseline_per_plane)

baseline_fid = None
if ENABLE_FID and _fid_metric is not None and fid_real_tensors:
    bl_fake = [load_image(r['ctx_paths'][-1]) for r in fid_rows]
    baseline_fid = compute_fid(fid_real_tensors, bl_fake)
    print(f'Baseline FID = {baseline_fid:.4f}' if baseline_fid is not None else 'Baseline FID = n/a')

# Copy-last-frame rollout baseline (frozen newest context frame across each horizon).
# Overall = mean over all (row, step) pairs, matching how model rollout overall is computed.
baseline_rollout_overall: dict[str, float] = {}
for _h, _h_rows in rollout_rows_by_horizon.items():
    _bl_recs = eval_copy_last_rollout_rows(_h_rows, steps=_h, desc=f'copy-last rollout H={_h}')
    _bl_odf  = summarize_records(_bl_recs, [])
    if not _bl_odf.empty:
        _bl_r = _bl_odf.to_dict('records')[0]
        for _m in ('mse', 'l1', 'ssim', 'lpips'):
            baseline_rollout_overall[f'rollout_h{_h}_{_m}'] = float(_bl_r.get(_m, float('nan')))


# ---- Main evaluation loop ----

def _safe_version(mod_name: str):
    try:
        return __import__(mod_name).__version__
    except Exception:
        return None

summary_payload  = {
    'eval_split': EVAL_SPLIT, 'run_prefix': RUN_PREFIX,
    'ddim_steps': DDIM_STEPS, 'ddim_eta': DDIM_ETA,
    'cfg_guidance_scale': CFG_GUIDANCE_SCALE,
    'use_residual_prediction': bool(USE_RESIDUAL_PREDICTION),
    'trained_epoch': int(_primary_ckpt.get('epoch')) if isinstance(_primary_ckpt, dict) and _primary_ckpt.get('epoch') is not None else None,
    'eval_config': {
        'one_step_samples_per_plane': ONE_STEP_SAMPLES_PER_PLANE,
        'rollout_horizons': ROLLOUT_HORIZONS,
        'rollout_samples_per_horizon': ROLLOUT_SAMPLES_PER_HORIZON,
        'fid_samples': FID_SAMPLES if ENABLE_FID else 0,
        'eval_batch_size': EVAL_BATCH_SIZE,
    },
    'library_versions': {m: _safe_version(m) for m in ('torch', 'torchvision', 'diffusers', 'torchmetrics', 'lpips')},
    'baseline': baseline_overall_df.to_dict('records')[0] if not baseline_overall_df.empty else None,
    'models': {},
}

per_plane_tables, per_phase_tables, transition_tables = [], [], []
rollout_tables, summary_rows = [], []

_bl = baseline_overall_df.to_dict('records')[0] if not baseline_overall_df.empty else {}
summary_rows.append({
    'model': 'copy_last_frame', 'checkpoint': 'N/A',
    'one_step_mse':   float(_bl.get('mse',   float('nan'))),
    'one_step_l1':    float(_bl.get('l1',    float('nan'))),
    'one_step_psnr':  float(_bl.get('psnr',  float('nan'))),
    'one_step_ssim':  float(_bl.get('ssim',  float('nan'))),
    'one_step_lpips': float(_bl.get('lpips', float('nan')) if _bl.get('lpips') is not None else float('nan')),
    'one_step_fid':   float(baseline_fid) if baseline_fid is not None else float('nan'),
    **{f'rollout_h{h}_{m}': baseline_rollout_overall.get(f'rollout_h{h}_{m}', float('nan'))
       for h in ROLLOUT_HORIZONS for m in ('mse', 'l1', 'ssim', 'lpips')},
})


for model_cfg in MODEL_CKPTS:
    model_name = model_cfg['name']
    ckpt_path  = model_cfg['path']
    activate_checkpoint(model_name, ckpt_path)

    print(f'\n=== Evaluating {model_name} ===')

    # One-step metrics
    one_step_records, one_step_gen = eval_one_step_rows(
        one_step_rows, desc=f'{model_name} one-step', return_generated=True,
    )
    transition_records     = eval_one_step_rows(transition_rows,     desc=f'{model_name} transition')
    non_transition_records = eval_one_step_rows(non_transition_rows, desc=f'{model_name} non-transition')

    # FID (batched sampling — far faster than the previous per-row loop)
    model_fid = None
    if ENABLE_FID and _fid_metric is not None and fid_real_tensors:
        fid_gen = list(sample_rows_batched(fid_rows, desc=f'{model_name} FID sampling'))
        model_fid = compute_fid(fid_real_tensors, fid_gen)
        print(f'FID = {model_fid:.4f}' if model_fid is not None else 'FID = n/a')

    # Stratified summaries
    per_plane_df = summarize_records(one_step_records, ['plane'])
    per_phase_df = summarize_records(one_step_records, ['phase_name'])
    trans_all    = (
        [dict(r, subset='transition')     for r in transition_records] +
        [dict(r, subset='non_transition') for r in non_transition_records]
    )
    trans_df = summarize_records(trans_all, ['subset'])

    for df, lst, col_name in [
        (per_plane_df, per_plane_tables, 'plane'),
        (per_phase_df, per_phase_tables, 'phase_name'),
        (trans_df,     transition_tables, 'subset'),
    ]:
        if not df.empty:
            df.insert(0, 'model', model_name)
            lst.append(df)

    # Rollout evaluation
    rollout_model_frames = []
    for h, h_rows in rollout_rows_by_horizon.items():
        roll_recs = eval_rollout_rows(h_rows, steps=h, desc=f'{model_name} rollout H={h}')
        if not roll_recs:
            continue
        step_df = summarize_records(roll_recs, ['horizon', 'step'])
        step_df['aggregation'] = 'per_step'
        all_df  = summarize_records(roll_recs, ['horizon'])
        if not all_df.empty:
            all_df['step'] = 'all'; all_df['aggregation'] = 'overall'
        combined = pd.concat([step_df, all_df], ignore_index=True, sort=False)
        combined.insert(0, 'model', model_name)
        rollout_model_frames.append(combined)
        rollout_tables.append(combined)

    # Qualitative grids
    if SAVE_QUALITATIVE:
        model_root   = QUALITATIVE_ROOT / model_name
        one_step_dir = model_root / 'one_step'
        one_step_dir.mkdir(parents=True, exist_ok=True)
        _save_sample_grid(
            one_step_rows[:QUALITATIVE_SAMPLES], one_step_dir,
            title_suffix='(evaluation one-step)',
            precomputed_gen=one_step_gen[:QUALITATIVE_SAMPLES],
        )
        for h in QUALITATIVE_ROLLOUT_HORIZONS:
            roll_dir = model_root / f'rollout_h{h}'
            roll_dir.mkdir(parents=True, exist_ok=True)
            for row in rollout_rows_by_horizon.get(h, [])[:QUALITATIVE_ROLLOUT_SAMPLES]:
                _save_rollout_grid(row, roll_dir, steps=h)

    # Build comparison row
    overall_df = summarize_records(one_step_records, [])
    comp = {
        'model': model_name, 'checkpoint': ckpt_path.name,
        'one_step_mse':   float(overall_df.iloc[0]['mse'])  if not overall_df.empty else float('nan'),
        'one_step_l1':    float(overall_df.iloc[0]['l1'])   if not overall_df.empty else float('nan'),
        'one_step_psnr':  float(overall_df.iloc[0]['psnr']) if not overall_df.empty else float('nan'),
        'one_step_ssim':  float(overall_df.iloc[0]['ssim']) if not overall_df.empty else float('nan'),
        'one_step_lpips': float(overall_df.iloc[0]['lpips'] if (not overall_df.empty and overall_df.iloc[0]['lpips'] is not None) else float('nan')),
        'one_step_fid':   float(model_fid) if model_fid is not None else float('nan'),
    }
    if rollout_model_frames:
        rm_df = pd.concat(rollout_model_frames, ignore_index=True, sort=False)
        for h in ROLLOUT_HORIZONS:
            mask = (rm_df['horizon'] == h) & (rm_df['aggregation'] == 'overall')
            if mask.any():
                r = rm_df[mask].iloc[0]
                for met in ('mse', 'l1', 'ssim', 'lpips'):
                    comp[f'rollout_h{h}_{met}'] = float(r.get(met, float('nan')))
            else:
                for met in ('mse', 'l1', 'ssim', 'lpips'):
                    comp[f'rollout_h{h}_{met}'] = float('nan')
    else:
        for h in ROLLOUT_HORIZONS:
            for met in ('mse', 'l1', 'ssim', 'lpips'):
                comp[f'rollout_h{h}_{met}'] = float('nan')

    summary_rows.append(comp)
    summary_payload['models'][model_name] = {
        'checkpoint':        str(ckpt_path),
        'one_step_overall':  overall_df.to_dict('records')[0] if not overall_df.empty else None,
        'per_plane':         per_plane_df.to_dict('records'),
        'per_phase':         per_phase_df.to_dict('records'),
        'transition_metrics': trans_df.to_dict('records'),
        'fid':               float(model_fid) if model_fid is not None else None,
    }

    unload_active_model()

# ---- Persist all results ----

per_plane_df_all  = pd.concat(per_plane_tables,  ignore_index=True, sort=False) if per_plane_tables  else pd.DataFrame()
per_phase_df_all  = pd.concat(per_phase_tables,  ignore_index=True, sort=False) if per_phase_tables  else pd.DataFrame()
transition_df_all = pd.concat(transition_tables, ignore_index=True, sort=False) if transition_tables else pd.DataFrame()
rollout_df_all    = pd.concat(rollout_tables,     ignore_index=True, sort=False) if rollout_tables    else pd.DataFrame()
summary_df        = pd.DataFrame(summary_rows)

with open(METRICS_SUMMARY_PATH, 'w', encoding='utf-8') as _f:
    json.dump(summary_payload, _f, indent=2)
for df, path in [
    (per_plane_df_all,  PER_PLANE_CSV_PATH),
    (per_phase_df_all,  PER_PHASE_CSV_PATH),
    (transition_df_all, TRANSITION_CSV_PATH),
    (rollout_df_all,    ROLLOUT_CSV_PATH),
    (summary_df,        SUMMARY_CSV_PATH),
]:
    if not df.empty:
        df.to_csv(path, index=False)

print('\nSaved artifacts:')
for df, path in [
    (per_plane_df_all,  PER_PLANE_CSV_PATH),
    (per_phase_df_all,  PER_PHASE_CSV_PATH),
    (transition_df_all, TRANSITION_CSV_PATH),
    (rollout_df_all,    ROLLOUT_CSV_PATH),
    (summary_df,        SUMMARY_CSV_PATH),
]:
    print(' -', path.name if not df.empty else f'{path.name} (skipped, empty)')


One-step rows: 200
Transition rows: 100
Non-transition rows: 100
Rollout H=3: 30 rows
Rollout H=5: 30 rows
Rollout H=10: 30 rows
FID rows: 4096 (FID unreliable below ~2048 samples)

=== Copy-last-frame baseline ===


copy-last-frame baseline: 100%|██████████ 200/200 [00:00<00:00, 370.34it/s]

Baseline overall:


,n,mse,mse_se,l1,l1_se,psnr,psnr_se,ssim,ssim_se,lpips,lpips_se
0,200,0.060784,0.003782,0.136927,0.004568,19.553597,0.252959,0.644158,0.011401,0.039388,0.002104


Baseline per-plane:


...

Activated [best_val]: nextframe_ctxk4_128px_u32-64-128-256-256_lpb1_vdm_resid_best.pt
  Parameters: 20,689,714 | TemporalAttnBlocks: 10

=== Evaluating best_val ===


best_val one-step: 100%|██████████ 13/13 [02:07<00:00,  9.82s/it]
best_val transition: 100%|██████████ 7/7 [01:07<00:00,  9.66s/it]
best_val non-transition: 100%|██████████ 7/7 [01:06<00:00,  9.43s/it]
best_val FID sampling: 100%|██████████ 256/256 [42:41<00:00, 10.00s/it]


FID = 3.2498


best_val rollout H=3: 100%|██████████ 2/2 [00:56<00:00, 28.26s/it]
best_val rollout H=5: 100%|██████████ 2/2 [01:32<00:00, 46.24s/it]
best_val rollout H=10: 100%|██████████ 2/2 [03:06<00:00, 93.48s/it]



Saved artifacts:
 - per_plane.csv
 - per_phase.csv
 - transition_metrics.csv
 - rollout_by_horizon.csv
 - summary.csv


In [15]:
from IPython.display import display as _disp, HTML as _HTML

def _render_table(df, title: str):
    if df is None or df.empty:
        print(f'{title}: (no data)')
        return
    _disp(_HTML(f'<h3 style="margin-top:1.2em">{title}</h3>'))
    _disp(df.reset_index(drop=True).style.format(precision=4))

# One-step comparison
_os_cols = ['model', 'checkpoint', 'one_step_mse', 'one_step_l1', 'one_step_psnr', 'one_step_ssim', 'one_step_lpips', 'one_step_fid']
_render_table(summary_df[[c for c in _os_cols if c in summary_df.columns]], 'One-step prediction summary')

# Per-plane breakdown
_render_table(per_plane_df_all, 'Per-plane metrics (one-step)')

# Per-phase breakdown
_render_table(per_phase_df_all, 'Per-phase metrics (one-step)')

# Transition vs non-transition
_render_table(transition_df_all, 'Transition vs non-transition (one-step)')

# Rollout tables — one per horizon
for _h in ROLLOUT_HORIZONS:
    _r_cols = ['model', 'checkpoint'] + [f'rollout_h{int(_h)}_{m}' for m in ('mse', 'l1', 'ssim', 'lpips')]
    _render_table(
        summary_df[[c for c in _r_cols if c in summary_df.columns]],
        f'Rollout H={_h} (aggregated over all steps)',
    )

# Full rollout per-step breakdown
if not rollout_df_all.empty:
    for _h in ROLLOUT_HORIZONS:
        mask = (rollout_df_all['horizon'] == _h) & (rollout_df_all.get('aggregation', '') == 'per_step')
        _render_table(rollout_df_all[mask] if mask.any() else pd.DataFrame(), f'Rollout H={_h} per step')

# ---- Model vs copy-last-frame verdict (one-step) ----
# Directly answers "does the model beat the trivial copy-last-frame predictor?"
# Lower-is-better: mse, l1, lpips, fid.  Higher-is-better: psnr, ssim.
if not summary_df.empty and 'copy_last_frame' in set(summary_df['model']):
    _bl_row = summary_df[summary_df['model'] == 'copy_last_frame'].iloc[0]
    _lower_is_better = {'one_step_mse', 'one_step_l1', 'one_step_lpips', 'one_step_fid'}
    _verdict_rows = []
    for _, _mrow in summary_df[summary_df['model'] != 'copy_last_frame'].iterrows():
        for _metric in ['one_step_mse', 'one_step_l1', 'one_step_psnr',
                        'one_step_ssim', 'one_step_lpips', 'one_step_fid']:
            _mv, _bv = _mrow.get(_metric), _bl_row.get(_metric)
            if _mv is None or _bv is None or pd.isna(_mv) or pd.isna(_bv):
                continue
            _better = (_mv < _bv) if _metric in _lower_is_better else (_mv > _bv)
            _verdict_rows.append({
                'model':          _mrow['model'],
                'metric':         _metric.replace('one_step_', ''),
                'model_value':    float(_mv),
                'baseline_value': float(_bv),
                'delta':          float(_mv - _bv),
                'model_better':   'yes' if _better else 'no',
            })
    _verdict_df = pd.DataFrame(_verdict_rows)
    _render_table(_verdict_df, 'Model vs copy-last-frame (one-step) — does the model beat the trivial copy?')
    if not _verdict_df.empty:
        _verdict_df.to_csv(ARTIFACT_ROOT / 'baseline_verdict.csv', index=False)
        _n_better = int((_verdict_df['model_better'] == 'yes').sum())
        print(f'Model beats copy-last-frame on {_n_better}/{len(_verdict_df)} one-step metrics '
              f'(split={EVAL_SPLIT}). Saved -> {(ARTIFACT_ROOT / "baseline_verdict.csv").name}')


,model,checkpoint,one_step_mse,one_step_l1,one_step_psnr,one_step_ssim,one_step_lpips,one_step_fid
0,copy_last_frame,N/A,0.0608,0.1369,19.5536,0.6442,0.0394,1.3553
1,best_val,nextframe_ctxk4_128px_u32-64-128-256-256_lpb1_vdm_resid_best.pt,0.0628,0.1403,19.2404,0.6363,0.0436,3.2498


,model,plane,n,mse,mse_se,l1,l1_se,psnr,psnr_se,ssim,ssim_se,lpips,lpips_se
0,best_val,F0,200,0.0628,0.0037,0.1403,0.0044,19.2404,0.2332,0.6363,0.0109,0.0436,0.0021


,model,phase_name,n,mse,mse_se,l1,l1_se,psnr,psnr_se,ssim,ssim_se,lpips,lpips_se
0,best_val,p2,13,0.0635,0.0089,0.1487,0.0120,18.5431,0.6591,0.5955,0.0354,0.0443,0.0073
1,best_val,p3,3,0.0362,0.0111,0.1018,0.0199,21.0277,1.7506,0.7342,0.0568,0.0353,0.0077
2,best_val,p4,24,0.0491,0.0078,0.1229,0.0108,20.3560,0.7282,0.6802,0.0311,0.0384,0.0036
3,best_val,p5,4,0.0597,0.0110,0.1466,0.0126,18.4533,0.7242,0.6316,0.0369,0.0463,0.0152
4,best_val,p6,6,0.0569,0.0117,0.1310,0.0152,19.0488,1.0637,0.6957,0.0436,0.0355,0.0061
5,best_val,p7,8,0.0458,0.0093,0.1224,0.0115,19.8560,0.6865,0.6521,0.0286,0.0474,0.0055
6,best_val,p8,32,0.0604,0.0073,0.1370,0.0094,19.2910,0.5852,0.6419,0.0241,0.0383,0.0028
7,best_val,p9+,33,0.0697,0.0110,0.1454,0.0126,18.8341,0.5653,0.6488,0.0296,0.0406,0.0032
8,best_val,pB,5,0.1226,0.0721,0.1967,0.0747,17.4438,2.0700,0.6078,0.1224,0.0708,0.0320
9,best_val,pEB,11,0.0965,0.0116,0.1881,0.0140,16.5979,0.6640,0.5322,0.0396,0.0618,0.0059


,model,subset,n,mse,mse_se,l1,l1_se,psnr,psnr_se,ssim,ssim_se,lpips,lpips_se
0,best_val,non_transition,100,0.0660,0.0054,0.1451,0.0066,19.0309,0.3331,0.6206,0.0158,0.0438,0.0021
1,best_val,transition,100,0.0642,0.0042,0.1424,0.0053,18.8718,0.3034,0.6229,0.0135,0.0495,0.0032


,model,checkpoint,rollout_h3_mse,rollout_h3_l1,rollout_h3_ssim,rollout_h3_lpips
0,copy_last_frame,N/A,0.0514,0.1252,0.6791,0.0363
1,best_val,nextframe_ctxk4_128px_u32-64-128-256-256_lpb1_vdm_resid_best.pt,0.0577,0.1337,0.6597,0.0415


...

,model,horizon,step,n,mse,mse_se,l1,l1_se,psnr,psnr_se,ssim,ssim_se,lpips,lpips_se,aggregation
0,best_val,3,1,30,0.0512,0.0062,0.1227,0.0087,20.0575,0.6258,0.6864,0.0227,0.0364,0.0025,per_step
1,best_val,3,2,30,0.0583,0.0063,0.1346,0.0086,19.3126,0.5868,0.6561,0.0240,0.0429,0.0033,per_step
2,best_val,3,3,30,0.0637,0.0059,0.1438,0.0072,18.4552,0.3718,0.6364,0.0190,0.0452,0.0026,per_step


,model,horizon,step,n,mse,mse_se,l1,l1_se,psnr,psnr_se,ssim,ssim_se,lpips,lpips_se,aggregation
0,best_val,5,1,30,0.0673,0.0082,0.1451,0.0111,18.9491,0.6717,0.6161,0.0299,0.0456,0.0036,per_step
1,best_val,5,2,30,0.0801,0.0081,0.1652,0.0099,17.6759,0.4789,0.5660,0.0253,0.0564,0.0041,per_step
2,best_val,5,3,30,0.0757,0.0067,0.1593,0.0085,17.7857,0.4326,0.5728,0.0236,0.0583,0.0038,per_step
3,best_val,5,4,30,0.0683,0.0055,0.1488,0.0067,18.0763,0.3526,0.6022,0.0188,0.0582,0.0034,per_step
4,best_val,5,5,30,0.0951,0.0095,0.1818,0.0115,16.8263,0.4191,0.5216,0.0261,0.0723,0.0063,per_step


,model,horizon,step,n,mse,mse_se,l1,l1_se,psnr,psnr_se,ssim,ssim_se,lpips,lpips_se,aggregation
0,best_val,10,1,30,0.0601,0.0104,0.1332,0.0126,19.9183,0.7292,0.6521,0.0288,0.0406,0.0034,per_step
1,best_val,10,2,30,0.0608,0.0067,0.1382,0.0084,18.9056,0.4700,0.6326,0.0217,0.0480,0.0035,per_step
2,best_val,10,3,30,0.0713,0.0089,0.1494,0.0104,18.6423,0.6592,0.6102,0.0248,0.0535,0.0046,per_step
3,best_val,10,4,30,0.0719,0.0079,0.1511,0.0101,18.5677,0.6802,0.6054,0.0252,0.0548,0.0038,per_step
4,best_val,10,5,30,0.0647,0.0081,0.1406,0.0103,18.9560,0.5820,0.6332,0.0249,0.0518,0.0037,per_step
5,best_val,10,6,30,0.0709,0.0071,0.1520,0.0088,18.2805,0.5123,0.6003,0.0222,0.0586,0.0040,per_step
6,best_val,10,7,30,0.0805,0.0085,0.1602,0.0100,17.9269,0.6163,0.5822,0.0246,0.0740,0.0113,per_step
7,best_val,10,8,30,0.0737,0.0080,0.1560,0.0099,18.1719,0.5338,0.5933,0.0253,0.0748,0.0113,per_step
8,best_val,10,9,30,0.0842,0.0089,0.1668,0.0110,17.7437,0.6080,0.5611,0.0269,0.0790,0.0110,per_step
9,best_val,10,10,30,0.0749,0.0079,0.1568,0.0103,18.1056,0.5384,0.5888,0.0265,0.0794,0.0111,per_step


,model,metric,model_value,baseline_value,delta,model_better
0,best_val,mse,0.0628,0.0608,0.0020,no
1,best_val,l1,0.1403,0.1369,0.0034,no
2,best_val,psnr,19.2404,19.5536,-0.3132,no
3,best_val,ssim,0.6363,0.6442,-0.0079,no
4,best_val,lpips,0.0436,0.0394,0.0043,no
5,best_val,fid,3.2498,1.3553,1.8945,no


Model beats copy-last-frame on 0/6 one-step metrics (split=test). Saved -> baseline_verdict.csv


In [16]:
# ==== CFG guidance-scale sweep (one-step, primary checkpoint) ====
# Finds the guidance scale that minimises one-step error. Reuses the SAME fixed
# `one_step_rows` and helpers as the main evaluation, so it must be run AFTER the
# main evaluation cell. The previous default CFG=3.0 over-sharpened these
# near-deterministic next-frame predictions; this sweep locates a better value and
# persists the winner so the next run's config cell wires it in automatically.

@torch.inference_mode()
def eval_one_step_cfg(rows: list[dict], cfg_scale: float, desc: str) -> list[dict]:
    records = []
    for start in tqdm_bar(range(0, len(rows), EVAL_BATCH_SIZE), desc=desc):
        batch = rows[start:start + EVAL_BATCH_SIZE]
        tensors = rows_to_tensors(batch)
        if USE_TIME_MAP:
            x_ctx, x_tgt, fi = tensors; fi = fi.to(device)
        else:
            x_ctx, x_tgt = tensors; fi = None
        x_ctx = x_ctx.to(device); x_tgt = x_tgt.to(device)
        x_gen = sample_from_model(
            (x_ctx.size(0), MODEL_OUT_CHANNELS, IMG_SIZE, IMG_SIZE),
            x_ctx=x_ctx, frame_idx=fi, cfg_scale=cfg_scale,
        )
        for i in range(len(batch)):
            records.append(metric_record(x_gen[i:i+1], x_tgt[i:i+1]))
    return records


# Sweep on the primary (best_val) checkpoint; fall back to the first discovered one.
_sweep_ckpt = next((mc for mc in MODEL_CKPTS if mc['name'] == 'best_val'), MODEL_CKPTS[0])
activate_checkpoint(_sweep_ckpt['name'], _sweep_ckpt['path'])

_sweep_rows = []
# Copy-last-frame baseline (CFG-independent reference point).
_bl_df = summarize_records(eval_copy_last_frame(one_step_rows, desc='copy-last baseline'), [])
if not _bl_df.empty:
    _r = _bl_df.to_dict('records')[0]
    _sweep_rows.append({'cfg_scale': 'copy_last', **{k: _r.get(k) for k in ('mse', 'l1', 'ssim', 'lpips')}})

for _cfg in EVAL_CFG_SCALES:
    _recs = eval_one_step_cfg(one_step_rows, _cfg, desc=f'{_sweep_ckpt["name"]} CFG={_cfg}')
    _df = summarize_records(_recs, [])
    if not _df.empty:
        _r = _df.to_dict('records')[0]
        _sweep_rows.append({'cfg_scale': _cfg, **{k: _r.get(k) for k in ('mse', 'l1', 'ssim', 'lpips')}})

unload_active_model()

cfg_sweep_df = pd.DataFrame(_sweep_rows)
cfg_sweep_df.to_csv(ARTIFACT_ROOT / 'cfg_sweep.csv', index=False)
print('Saved CFG sweep ->', ARTIFACT_ROOT / 'cfg_sweep.csv')

# Pick the best model CFG by LPIPS (fall back to MSE if LPIPS unavailable) and persist it
# next to the checkpoints, keyed by run prefix, so the config cell auto-wires it next run.
_model_only = cfg_sweep_df[cfg_sweep_df['cfg_scale'] != 'copy_last'].copy()
if not _model_only.empty:
    _key = 'lpips' if _model_only['lpips'].notna().any() else 'mse'
    _best = _model_only.loc[_model_only[_key].astype(float).idxmin()]
    _best_scale = float(_best['cfg_scale'])
    print(f"Best CFG by {_key}: {_best_scale}  "
          f"(mse={_best.get('mse'):.5f}, lpips={_best.get('lpips')})")
    _best_cfg_out = MODEL_DIR / f'{RUN_PREFIX}_best_cfg.json'
    _best_cfg_out.write_text(json.dumps({
        'best_cfg_scale': _best_scale,
        'selection_metric': _key,
        'eval_split': EVAL_SPLIT,
        'n_rows': int(len(one_step_rows)),
    }, indent=2))
    print(f'Persisted tuned CFG -> {_best_cfg_out.name} '
          f'(the config cell loads this automatically on the next run)')

try:
    from IPython.display import display as _disp2
    _disp2(cfg_sweep_df.reset_index(drop=True).style.format(precision=4))
except Exception:
    print(cfg_sweep_df)


Activated [best_val]: nextframe_ctxk4_128px_u32-64-128-256-256_lpb1_vdm_resid_best.pt
  Parameters: 20,689,714 | TemporalAttnBlocks: 10


copy-last baseline: 100%|██████████ 200/200 [00:00<00:00, 345.22it/s]
best_val CFG=1.0: 100%|██████████ 13/13 [02:01<00:00,  9.33s/it]
best_val CFG=1.5: 100%|██████████ 13/13 [04:04<00:00, 18.78s/it]
best_val CFG=2.0: 100%|██████████ 13/13 [04:04<00:00, 18.77s/it]
best_val CFG=3.0: 100%|██████████ 13/13 [04:02<00:00, 18.65s/it]

Saved CFG sweep -> /home/khasion/Projects/thesis/diffusion/diffusion_eval/test/cfg_sweep.csv
Best CFG by lpips: 1.0  (mse=0.06474, lpips=0.04398090017959475)
Persisted tuned CFG -> nextframe_ctxk4_128px_u32-64-128-256-256_lpb1_vdm_resid_best_cfg.json (the config cell loads this automatically on the next run)


,cfg_scale,mse,l1,ssim,lpips
0,copy_last,0.0608,0.1369,0.6442,0.0394
1,1.0000,0.0647,0.1430,0.6284,0.0440
2,1.5000,0.0729,0.1539,0.5917,0.0476
3,2.0000,0.0836,0.1683,0.5476,0.0511
4,3.0000,0.1004,0.1893,0.4889,0.0566
